# Lesson 1.2 — OS & Process Automation

**Level: Beginner**

## Learning Objectives
- Run shell commands safely with `subprocess.run()` and capture output
- Read and set environment variables via `os.environ`
- Navigate file paths with `pathlib.Path` (cross-platform)
- Copy, move, and inspect files with `shutil`
- Write a system health-check script

**Produces:** `system_health.py` — a reusable health-check script.

*Official refs:*
- *[subprocess — Subprocess management](https://docs.python.org/3/library/subprocess.html)*
- *[pathlib — Object-oriented filesystem paths](https://docs.python.org/3/library/pathlib.html)*

## Concepts

### `subprocess.run()` — the Right Way to Run Commands

The `subprocess` module is Python's gateway to the OS shell.  
Always prefer `subprocess.run()` over `os.system()` — it gives you full control
over stdin/stdout/stderr and lets you inspect the return code.

```python
# Official example (subprocess docs):
import subprocess
result = subprocess.run(["ls", "-l", "/dev/null"], capture_output=True)
# → CompletedProcess(args=['ls', ...], returncode=0, stdout=b'...', stderr=b'')
```

Key parameters from the [official docs](https://docs.python.org/3/library/subprocess.html#subprocess.run):

| Parameter | Effect |
|-----------|--------|
| `capture_output=True` | Sets `stdout=PIPE, stderr=PIPE` |
| `text=True` | Decodes bytes → str automatically |
| `check=True` | Raises `CalledProcessError` on non-zero exit |
| `timeout=30` | Kills the process after 30 s |

### `pathlib.Path` — Cross-Platform File Paths

From the [pathlib docs](https://docs.python.org/3/library/pathlib.html):

> *"Path classes are divided between pure paths, which provide purely
> computational operations without I/O, and concrete paths, which also
> provide I/O operations."*

The `/` operator builds paths:
```python
from pathlib import Path
p = Path('/etc') / 'init.d' / 'reboot'   # PosixPath('/etc/init.d/reboot')
```

**Security note (subprocess docs):** Using `shell=True` exposes your script to
shell injection. Always pass commands as a **list**, never a formatted string with
user input.

In [ ]:
import subprocess, sys, os, shutil, platform
from pathlib import Path

print(f"Platform : {sys.platform}")
print(f"Python   : {sys.version.split()[0]}")
print(f"cwd      : {Path.cwd()}")

## Code-Along: Step 1 — Safe Command Execution

In [ ]:
def run_command(cmd: list[str], timeout: int = 30) -> dict:
    """Run a shell command and return a structured result dict.

    Based on subprocess.run() — see:
    https://docs.python.org/3/library/subprocess.html#subprocess.run

    Args:
        cmd: Command as a list, e.g. ['git', 'status']
             NEVER pass a single string with shell=True unless you trust
             the entire string (subprocess security note).
        timeout: Seconds before killing the process.

    Returns:
        {'success': bool, 'stdout': str, 'stderr': str, 'returncode': int}
    """
    try:
        result = subprocess.run(
            cmd,
            capture_output=True,   # captures both stdout and stderr
            text=True,             # decodes bytes to str
            timeout=timeout,
        )
        return {
            "success":    result.returncode == 0,
            "stdout":     result.stdout.strip(),
            "stderr":     result.stderr.strip(),
            "returncode": result.returncode,
        }
    except subprocess.TimeoutExpired:
        return {"success": False, "stdout": "", "stderr": "Timed out", "returncode": -1}
    except FileNotFoundError as e:
        return {"success": False, "stdout": "", "stderr": str(e), "returncode": -1}

# Verify Python version via subprocess
r = run_command([sys.executable, "--version"])
print(f"success  : {r['success']}")
print(f"output   : {r['stdout']}")
print(f"returncode: {r['returncode']}")

## Code-Along: Step 2 — Environment Variables

In [ ]:
# os.environ behaves like a dict (PEP 3137)
# Reference: https://docs.python.org/3/library/os.html#os.environ

def require_env(name: str) -> str:
    """Return an env var or raise EnvironmentError if missing."""
    val = os.environ.get(name)
    if val is None:
        raise EnvironmentError(
            f"Required env var '{name}' is not set. "
            "Check your .env file or CI/CD secrets configuration."
        )
    return val

def optional_env(name: str, default: str = "") -> str:
    return os.environ.get(name, default)

# Read common DevOps env vars
region = optional_env("AWS_DEFAULT_REGION", "us-east-1")
log_level = optional_env("LOG_LEVEL", "INFO")
env_name  = optional_env("ENVIRONMENT", "development")

print(f"AWS Region  : {region}")
print(f"Log Level   : {log_level}")
print(f"Environment : {env_name}")

# Set a var for this process only
os.environ["DEMO_VAR"] = "ops-bot"
print(f"\nDEMO_VAR set to: {os.environ['DEMO_VAR']}")

## Code-Along: Step 3 — pathlib and shutil

From the [pathlib docs](https://docs.python.org/3/library/pathlib.html#basic-use):

```python
from pathlib import Path

p = Path('.')
[x for x in p.iterdir() if x.is_dir()]   # list subdirectories
list(p.glob('**/*.py'))                   # recursive glob
```


In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    workspace = Path(tmp)

    # Build a directory tree
    (workspace / "configs" / "dev").mkdir(parents=True)
    (workspace / "configs" / "prod").mkdir(parents=True)
    (workspace / "logs").mkdir()

    # Write files
    dev_cfg  = workspace / "configs" / "dev"  / "app.yaml"
    prod_cfg = workspace / "configs" / "prod" / "app.yaml"
    dev_cfg.write_text("environment: development\nlog_level: DEBUG\nreplicas: 1\n")
    prod_cfg.write_text("environment: production\nlog_level: WARNING\nreplicas: 3\n")

    # Recursive listing (pathlib.rglob)
    print("Workspace tree:")
    for f in sorted(workspace.rglob("*")):
        rel = f.relative_to(workspace)
        indent = "  " * (len(rel.parts) - 1)
        icon = "📁" if f.is_dir() else "📄"
        print(f"  {indent}{icon} {rel.name}")

    # Read + stat
    content = dev_cfg.read_text()
    print(f"\ndev config ({dev_cfg.stat().st_size} bytes):\n{content}")

    # shutil.copy2 — preserves metadata
    backup = workspace / "configs" / "dev" / "app.yaml.bak"
    shutil.copy2(dev_cfg, backup)
    print(f"Backup created: {backup.name}")

## Code-Along: Step 4 — System Health Check Script

In [ ]:
def check_disk(path: str = "/") -> dict:
    """Return disk usage for a path using shutil.disk_usage."""
    if sys.platform == "win32":
        path = "C:\\"
    usage = shutil.disk_usage(path)
    pct = usage.used / usage.total * 100
    return {
        "path": path,
        "total_gb":    round(usage.total / 1024**3, 1),
        "used_gb":     round(usage.used  / 1024**3, 1),
        "free_gb":     round(usage.free  / 1024**3, 1),
        "percent_used": round(pct, 1),
        "healthy":     pct < 80,
    }

def tool_available(name: str) -> bool:
    """Return True if a CLI tool exists on PATH."""
    which = "where" if sys.platform == "win32" else "which"
    return run_command([which, name])["success"]

disk = check_disk()
print("=== System Health Report ===")
print(f"Host     : {platform.node()}")
print(f"OS       : {platform.system()} {platform.release()}")
print(f"Python   : {platform.python_version()}")
print()
print(f"Disk ({disk['path']}): {disk['used_gb']}GB / {disk['total_gb']}GB "
      f"({disk['percent_used']}%) — {'✓ Healthy' if disk['healthy'] else '⚠ >80% used'}")
print()
print("DevOps tools:")
for tool in ["git", "docker", "kubectl", "helm", "terraform"]:
    icon = "✓" if tool_available(tool) else "✗"
    print(f"  {icon}  {tool}")
print("=========================

## Exercise

Extend the health check to detect open ports.

**Your task:**
1. Write `check_port(host: str, port: int) -> bool` using `socket.create_connection`
2. Check ports 80, 443, and 8080 on `localhost`
3. Print `OPEN` or `CLOSED` for each

In [ ]:
import socket

# TODO: implement check_port(host, port) -> bool
# Hint: use socket.create_connection((host, port), timeout=1) inside try/except

def check_port(host: str, port: int) -> bool:
    pass   # ← your implementation here

# TODO: test it
for port in [80, 443, 8080]:
    status = "OPEN" if check_port("localhost", port) else "CLOSED"
    print(f"  localhost:{port:5}  {status}")


## Key Takeaways
- Pass commands as **lists** to `subprocess.run()` — never use `shell=True` with user input ([security note](https://docs.python.org/3/library/subprocess.html#security-considerations))
- `pathlib.Path` works identically on Windows, macOS, and Linux — prefer it over `os.path` string manipulation
- `shutil.disk_usage()` returns a quick disk health check without any external tools
- Health-check scripts like this become building blocks for your Prometheus exporter in Phase 5

---
**Next:** [Lesson 1.3 — Configuration Management](lesson-1.3-configuration-management.ipynb)